In [1]:
from dataAnalysis.DataAnalysis import DataAnalysis
import pandas as pd
from torch.nn.functional import normalize
from sklearn.ensemble import RandomForestClassifier
from EnsembleFramework import Framework
from sklearn.metrics import roc_curve
from numpy import sqrt, argmax
from joblib import dump, load
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import shap
import os
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

In [2]:
from pylab import *
from math import sin
rc('text', usetex = False)
la = matplotlib.font_manager.FontManager()
lu = matplotlib.font_manager.FontProperties(family = 'Arial')

In [3]:
data = pd.read_csv(r"extdata/sbcdata.csv", header=0)
data_analysis = DataAnalysis(data)

/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [4]:
sorted_train_data = data_analysis.get_training_data().sort_values(by="Id").reset_index(drop = True)
train_df = sorted_train_data.loc[:sorted_train_data.shape[0]*.8,:]
val_df = sorted_train_data.loc[sorted_train_data.shape[0]*.8:,:]
test_df = data_analysis.get_testing_data()
gw_df = data_analysis.get_gw_testing_data()

In [5]:
import numpy as np
import torch
from dataAnalysis.Constants import FEATURES, LABEL_COLUMN_NAME

def convert_cat_feature(df):
    df = df.copy()
    df["SexCategory"] = (df["Sex"] == "W").astype(int)
    return df
    
def get_graph(df):
    edge_index = []
    df = convert_cat_feature(df)
    df = df.sort_values(by=["Id", "Time"]).reset_index(drop=True)
    
    ## group df by ids
    for identifier, group in df.groupby("Id"):
        offset = group.index[0]
        triu_matrix = np.triu((group.index.values + np.identity(1))[0])
        triu_exp_matrix = np.expand_dims(triu_matrix, axis=-1)
    
        idx_shape = group.index.shape[0]
        idx_matrix = np.ones((idx_shape, idx_shape)) * np.arange(idx_shape) + 1 + offset
        idx_matrix = np.transpose(idx_matrix)
        idx_exp_matrix = np.expand_dims(idx_matrix, axis=-1)
    
        unprocess_edges = np.concatenate((idx_exp_matrix, triu_exp_matrix), axis=-1)
        reshaped_unprocess_edges = np.reshape(unprocess_edges, (-1, 2))
        mask = (reshaped_unprocess_edges[:, 0] * reshaped_unprocess_edges[:, 1]) != 0
        edge_index.append((reshaped_unprocess_edges[mask] - 1).astype(np.int64))
    edge_index_torch = torch.from_numpy(np.concatenate(edge_index)).type(torch.long).transpose(0,1)
    features_torch = torch.from_numpy(df[FEATURES].values).type(torch.float)
    labels_torch = torch.from_numpy((df.sort_values(by=["Id", "Time"])[LABEL_COLUMN_NAME] == "Sepsis").values).type(torch.long)
    return features_torch, edge_index_torch, labels_torch

In [6]:
X_train_comp, edge_index_train_comp, y_train_comp = get_graph(sorted_train_data)
X_train, edge_index_train, y_train = get_graph(train_df)
X_val, edge_index_val, y_val = get_graph(val_df)
X_test, edge_index_test, y_test = get_graph(test_df)
X_gw, edge_index_gw, y_gw = get_graph(gw_df)

In [7]:
control_nodes = X_train_comp[y_train_comp == 0]
median_ref_node = torch.median(control_nodes, dim = 0)[0]
mean_ref_node = torch.mean(control_nodes, dim = 0)

In [8]:
median_ref_node, mean_ref_node

(tensor([ 59.0000,   1.0000,   7.8000,   7.4000,   4.2300,  87.7000, 236.0000]),
 tensor([ 56.7933,   0.5038,   7.5724,   8.2361,   4.1218,  87.7919, 243.2722]))

In [9]:
def get_features(framework, X, edge_index, mask):
    return framework.get_features(X, edge_index, mask)

In [10]:
def append_ref_node(X, edge_index, ref_node):
    X= torch.clone(X)
    X_new = torch.cat([X, ref_node.unsqueeze(0)], dim = 0)
    mask = torch.ones(X_new.shape[0], dtype= torch.bool)
    mask[-1] = False
    ref_target_nodes = torch.arange(X_new.shape[0])
    ref_source_nodes = torch.ones_like(ref_target_nodes) * (X_new.shape[0] -1)
    ref_edge_index = torch.stack([ref_source_nodes, ref_target_nodes])
    edge_index_new = torch.cat([edge_index, ref_edge_index], dim = -1)
    return X_new, edge_index_new, mask

In [11]:
def get_sets_dict(ref_node, 
                     train_comp_edge_index,
                     train_edge_index,
                     val_edge_index,
                     test_edge_index,
                     gw_edge_index,
                    ):
    global X_train_comp, X_train, X_val, X_test, X_gw, y_train_comp, y_train, y_val, y_test, y_gw
    X_train_comp_new, edge_index_train_comp, mask_train_comp = append_ref_node(X_train_comp, train_comp_edge_index, ref_node)
    X_train_new, edge_index_train, mask_train = append_ref_node(X_train, train_edge_index, ref_node)
    X_val_new, edge_index_val, mask_val = append_ref_node(X_val, val_edge_index, ref_node)
    X_test_new, edge_index_test, mask_test = append_ref_node(X_test, test_edge_index, ref_node)
    X_gw_new, edge_index_gw, mask_gw = append_ref_node(X_gw, gw_edge_index, ref_node)
    sets = [("train_comp", X_train_comp_new, edge_index_train_comp, mask_train_comp, y_train_comp), ("train", X_train_new, edge_index_train,mask_train, y_train),
                ("val", X_val_new, edge_index_val,mask_val, y_val), ("test", X_test_new, edge_index_test,mask_test, y_test),  ("gw", X_gw_new, edge_index_gw,mask_gw, y_gw)]
    sets_dict = {graph_set[0]: (graph_set[1:]) for graph_set in sets}
    return sets_dict

In [12]:
median_dir_set_dict = get_sets_dict(median_ref_node, edge_index_train_comp, edge_index_train, edge_index_val, edge_index_test, edge_index_gw)
mean_dir_set_dict = get_sets_dict(mean_ref_node, edge_index_train_comp, edge_index_train, edge_index_val, edge_index_test, edge_index_gw)

In [13]:
def diff_user_function(kwargs):
    return kwargs["original_features"] - kwargs["mean_neighbors"]

def div_user_function(kwargs):
    return kwargs["original_features"] / kwargs["mean_neighbors"]

user_functions = {
    "diff.": diff_user_function,
    "quot.": div_user_function
}
hops = [0, 1]

In [21]:
def test_auroc_and_auprc(framework, clf, X, edge_index,mask, y, sc = None):
    features = torch.cat(get_features(framework, X, edge_index, mask), dim = 1).cpu().numpy()
    if sc is not None:
        features = sc.transform(features)
    pred_proba = clf.predict_proba(features)[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    return auroc, auprc, pred_proba

In [22]:
edge_type_sets = {
    "mean": mean_dir_set_dict,
    "median": median_dir_set_dict,
}

In [23]:
def fit_prospective_model(prospective_model, edge_type, user_function, sc = None):
    framework = Framework(user_functions=[user_function, user_function], 
                             hops_list=[0,1],
                             clfs=[None, None],
                             gpu_idx=0,
                             handle_nan=0.0,
                            attention_configs=[ None, None])
    X_train, edge_index_train, mask, y_train = edge_type_sets[edge_type]["train_comp"]
    features_train = torch.cat(get_features(framework, X_train, edge_index_train, mask), dim = 1).cpu().numpy()
    if sc is not None:
        sc.fit(features_train)
        features_train = sc.transform(features_train)
    prospective_model.fit(features_train, y_train)
    return prospective_model, framework

def evaluate_prospective_model(prospective_model, framework,edge_type, sc = None):
    eval_dict = dict()
    pred_probas_dict = dict()
    for name in edge_type_sets[edge_type]:
        X, edge_index,mask, y = edge_type_sets[edge_type][name]
        auroc, auprc, pred_proba = test_auroc_and_auprc(framework, prospective_model, X, edge_index,mask, y, sc)
        eval_dict[name] = dict()
        eval_dict[name]["auroc"] = np.round(auroc, 3)
        eval_dict[name]["auprc"] = np.round(auprc, 3)
        pred_probas_dict[name] = pred_proba
    return eval_dict, pred_probas_dict
    
def get_best_prospective_threshold(model, framework, edge_type, sc = None):
    X_test, edge_index_test, mask, y_test = edge_type_sets[edge_type]["test"]
    features_test = torch.cat(get_features(framework, X_test, edge_index_test, mask), dim = 1).cpu().numpy()
    if sc is not None:
        features_test = sc.transform(features_test)
    pred_proba = model.predict_proba(features_test)[:,1]
    fpr, tpr, thresholds = roc_curve(y_test, pred_proba)
    gmeans = sqrt(tpr * (1-fpr))
    ix = argmax(gmeans)
    return round(thresholds[ix], 4) 


def get_propsective_sensistivity_at_specifity(model, framework, edge_type, sc = None, desired_specificity = .8):
    sensitivity_dict = dict()
    for name in edge_type_sets[edge_type]:
        X, edge_index, mask, y = edge_type_sets[edge_type][name]
        features = torch.cat(get_features(framework, X, edge_index, mask), dim = 1).cpu().numpy()
        if sc is not None:
            features = sc.transform(features)
        y_pred_proba = model.predict_proba(features)[:, 1]
        fpr, tpr, thresholds = roc_curve(y, y_pred_proba)
        
        desired_specificity = 0.8
        desired_fpr = 1 - desired_specificity
        
        for i, fp_rate in enumerate(fpr):
            if fp_rate >= desired_fpr:
                threshold = thresholds[i]
                break
        
        y_pred = (y_pred_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
        sensitivity = tp / (tp + fn)
        sensitivity_dict[name] = {
            "sensitivity": sensitivity,
        }
    return sensitivity_dict

In [24]:
def print_and_export_shap_overview(shap_vals, title:str, overall_plt_title):

    samples_shap_vals = shap_vals[:, :7]
    time_shap_vals = shap_vals[:, 7:]

    plt.rcParams.update({'font.size': 12})
    fig =  plt.figure(figsize=(30, 15))
    gs = fig.add_gridspec(2, 2)
    FEATURES[1] = "Sex"
    
    ax = fig.add_subplot(gs[0, 0])
    plt.sca(ax)
    shap.summary_plot(samples_shap_vals, plot_type="bar", class_inds='original',show=False,  feature_names = FEATURES)
    ax.set_xlabel("Mean |shap values|")
    ax.grid()
    is_decision_tree = "_dt" in title
    if is_decision_tree:
        ax.tick_params(axis='x', labelsize=8)
    ax.set_title(f"Feature importance of original features")
    
    ax = fig.add_subplot(gs[0, 1])
    plt.sca(ax)
    shap.summary_plot(time_shap_vals, plot_type="bar",  class_inds='original',show=False,  feature_names = FEATURES)
    ax.set_xlabel("Mean |shap values|")
    ax.grid()
    ax.set_title(f"Feature importance of time features")
    
    ax = fig.add_subplot(gs[1, :])
    plt.sca(ax)
    shap.summary_plot(np.sum((np.abs(samples_shap_vals), np.abs(time_shap_vals)), axis = 0), plot_type="bar", show=False,  class_inds='original', feature_names = FEATURES)
    ax.set_xlabel("Mean |shap values|")
    ax.set_title(f"Overall feature importance")
    ax.grid()
    plt.tight_layout()
    plt.show()
    fig.savefig(f"{title}.png", dpi = 300, bbox_inches='tight',format="png")

    plt.rcParams.update({'font.size': 22})
    fig =  plt.figure(figsize=(30, 15))
    gs = fig.add_gridspec(2, 2)
    FEATURES[1] = "Sex"
    ax = fig.add_subplot(gs[:, :])
    plt.sca(ax)
    shap.summary_plot(np.sum((np.abs(samples_shap_vals), np.abs(time_shap_vals)), axis = 0), plot_type="bar", show=False,  class_inds='original', feature_names = FEATURES)
    ax.set_xlabel("Mean |shap values|")
    ax.set_title(overall_plt_title)
    ax.grid()
    plt.tight_layout()
    plt.show()
    fig.savefig(f"overall_{title}.png", dpi = 300, bbox_inches='tight',format="png")

In [33]:
def train_and_evaluate_models(ClassifierClass, config, name, only_return_pred_proba = False):
    pred_proba_dict = dict()
    for edge_type_set_key in edge_type_sets:
        pred_proba_dict[edge_type_set_key] = dict()
        for user_function_name in user_functions:
            sc = StandardScaler() if ClassifierClass.__name__ == "LogisticRegression" else None
            print(f"{name}_{edge_type_set_key}_{user_function_name}")
            model = ClassifierClass(**config[edge_type_set_key][user_function_name])
            user_function = user_functions[user_function_name]
            prospective_model, framework_dir = fit_prospective_model(model, edge_type_set_key, user_function, sc)
            
            eval_dict, pred_probas_dict = evaluate_prospective_model(prospective_model, framework_dir, edge_type_set_key,sc)
            
            display(HTML(pd.DataFrame(eval_dict).to_html()))
            best_threshold = get_best_prospective_threshold(prospective_model, framework_dir, edge_type_set_key, sc)
            print(f"Best threshold is {best_threshold}")
            sensitivity_scores = get_propsective_sensistivity_at_specifity(prospective_model, framework_dir, edge_type_set_key, sc)
            print("Sensitivity for 80% specifity")
            display(HTML(pd.DataFrame(sensitivity_scores).to_html()))
            model_file_name = f'prospective_{name}_{edge_type_set_key}_{user_function_name}'
            model_object_file_name = f'{model_file_name}.joblib'
            if not os.path.exists(model_object_file_name):
                dump(prospective_model, model_object_file_name)

            if only_return_pred_proba:
                pred_proba_dict[edge_type_set_key][user_function_name] = pred_probas_dict
                continue 
    
            X_train, edge_index_train, mask, _ = edge_type_sets[edge_type_set_key]["train"]
            background_data = torch.cat(get_features(framework_dir, X_train, edge_index_train, mask), dim = 1).cpu().numpy()
            if sc is not None:
                background_data = sc.transform(background_data)
            explainer = shap.LinearExplainer(prospective_model, background_data) if ClassifierClass.__name__ == "LogisticRegression" else shap.TreeExplainer(prospective_model)
            X_test, edge_index_test, mask, y_test = edge_type_sets[edge_type_set_key]["test"]
            features_test = torch.cat(get_features(framework_dir, X_test, edge_index_test, mask), dim = 1).cpu().numpy()
            if sc is not None:
                features_test = sc.transform(features_test)
                dump(sc, f'prospective_{name}_{edge_type_set_key}_{user_function_name}_sc.joblib')
            prospective_test_shap_vals = None
            if os.path.exists(f'{model_file_name}.npy'):
                prospective_test_shap_vals = np.load(f'{model_file_name}.npy')
            else:
                prospective_test_shap_vals = explainer.shap_values(features_test) #[-1]
                prospective_test_shap_vals = prospective_test_shap_vals[:, :, -1] if prospective_test_shap_vals.ndim == 3 else prospective_test_shap_vals  
                np.save(f'{model_file_name}.npy', prospective_test_shap_vals)
            print(prospective_test_shap_vals.shape)
            overall_output_file_name_dict = {
                "LogisticRegression": "Logistic Regression",
                "DecisionTreeClassifier": "Decision Tree",
                "RandomForestClassifier": "Random Forest",
                "XGBClassifier": "XGBoost"
            }            
            print_and_export_shap_overview(prospective_test_shap_vals, model_file_name, f"{overall_output_file_name_dict[ClassifierClass.__name__]} ({edge_type_set_key}, {user_function_name})")
    return pred_proba_dict

In [37]:
prospective_pred_probas_mean_diff_ref = pd.DataFrame()

prospective_pred_probas_mean_diff_ref["y"] = y_test

## Random forest

In [34]:
random_forest_config = {
    "mean": {
        "diff.": {'class_weight': {0: 0.0022, 1: 1}, 'max_leaf_nodes': 96, 'min_samples_leaf': 0.00022806861688903475, 'min_samples_split': 0.007050018577465296, 'n_estimators': 400, 'n_jobs': -1, 'random_state': 42},
        "quot.": {'class_weight': {0: 0.0022, 1: 1}, 'max_leaf_nodes': 100, 'min_samples_leaf': 2.0527079471076694e-05, 'min_samples_split': 0.0031003958580011876, 'n_estimators': 200, 'n_jobs': -1, 'random_state': 42}
    },
    "median": {
        "diff.": {'class_weight': {0: 0.0022, 1: 1}, 'max_leaf_nodes': 100, 'min_samples_leaf': 3.4338968904937804e-05, 'min_samples_split': 0.006641453523808764, 'n_estimators': 400, 'n_jobs': -1, 'random_state': 42},
        "quot.": {'class_weight': {0: 0.0022, 1: 1}, 'max_leaf_nodes': 97, 'min_samples_leaf': 0.0005314608635255335, 'min_samples_split': 0.005135518988032092, 'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}
    }
}

In [35]:
rf_pred_proba = train_and_evaluate_models(RandomForestClassifier, random_forest_config, "random_forest", only_return_pred_proba= True)

random_forest_mean_diff.


,train_comp,train,val,test,gw
auroc,0.934,0.934,0.936,0.892,0.841
auprc,0.035,0.034,0.036,0.021,0.008


Best threshold is 0.3726
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.916121,0.916597,0.914373,0.818367,0.738839


random_forest_mean_quot.


...

,train_comp,train,val,test,gw
sensitivity,0.915465,0.916597,0.911315,0.802041,0.723214


random_forest_median_quot.


,train_comp,train,val,test,gw
auroc,0.931,0.931,0.931,0.888,0.836
auprc,0.034,0.033,0.036,0.021,0.010


Best threshold is 0.3408
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.916121,0.917431,0.911315,0.806122,0.716518


In [38]:
prospective_pred_probas_mean_diff_ref[RandomForestClassifier.__name__] = rf_pred_proba["mean"]["diff."]["test"]

## Decision tree

In [39]:
decision_tree_config = {
    "mean": {
        "diff.": {'class_weight': 'balanced', 'criterion': 'log_loss', 'max_depth': 50, 'max_features': 8, 'min_samples_leaf': 0.014856077120078505, 'min_samples_split': 0.054100564805004726, 'random_state': 42, 'splitter': 'best'},
        "quot.": {'class_weight': {0: 0.0015, 1: 1}, 'criterion': 'entropy', 'max_depth': 60, 'max_features': 14, 'min_samples_leaf': 0.005351227449668673, 'min_samples_split': 0.025075714921275194, 'random_state': 42, 'splitter': 'best'}
    },
    "median": {
        "diff.": {'class_weight': {0: 0.0022, 1: 1}, 'criterion': 'log_loss', 'max_depth': None, 'max_features': 14, 'min_samples_leaf': 0.0031246952111982093, 'min_samples_split': 0.04387312031028205, 'random_state': 42, 'splitter': 'best'},
        "quot.": {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 68, 'max_features': 14, 'min_samples_leaf': 0.002225895294036227, 'min_samples_split': 0.0353704607533091, 'random_state': 42, 'splitter': 'best'}
    }
}

In [40]:
dt_pred_proba = train_and_evaluate_models(DecisionTreeClassifier, decision_tree_config, "decision_tree", only_return_pred_proba= True)

decision_tree_mean_diff.


,train_comp,train,val,test,gw
auroc,0.864,0.864,0.864,0.842,0.790
auprc,0.129,0.129,0.128,0.126,0.093


Best threshold is 0.5644
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.78768,0.791493,0.7737,0.730612,0.669643


decision_tree_mean_quot.


...

,train_comp,train,val,test,gw
sensitivity,0.777195,0.780651,0.764526,0.74898,0.609375


decision_tree_median_quot.


,train_comp,train,val,test,gw
auroc,0.887,0.887,0.887,0.856,0.805
auprc,0.153,0.154,0.148,0.146,0.118


Best threshold is 0.5912
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.804063,0.803169,0.807339,0.777551,0.671875


In [41]:
prospective_pred_probas_mean_diff_ref[DecisionTreeClassifier.__name__] = dt_pred_proba["mean"]["diff."]["test"]

## Logistic Regression

In [43]:
logistic_regression_config = {
    "mean": {
        "diff.": {'C': 1204.9209633748742, 'class_weight': {0: 0.0022147379349917173, 1: 1}, 'l1_ratio': 0.5471950921901692, 'max_iter': 300, 'random_state': 42, 'solver': 'liblinear'},
        "quot.": {'C': 54.177579045605064, 'class_weight': 'balanced', 'l1_ratio': 0.650073750667717, 'max_iter': 650, 'random_state': 42, 'solver': 'saga'}
    },
    "median": {
        "diff.": {'C': 883.7371290527146, 'class_weight': {0: 0.0022147379349917173, 1: 1}, 'l1_ratio': 0.15461787547766692, 'max_iter': 700, 'random_state': 42, 'solver': 'liblinear'},
        "quot.": {'C': 4053.859293470675, 'class_weight': {0: 0.0022147379349917173, 1: 1}, 'l1_ratio': 0.9086116482940818, 'max_iter': 900, 'random_state': 42, 'solver': 'liblinear'}
    }
}

In [44]:
lr_pred_proba = train_and_evaluate_models(LogisticRegression, logistic_regression_config, "logistic_regression", only_return_pred_proba= True)

logistic_regression_mean_diff.


/home/dwalke/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


,train_comp,train,val,test,gw
auroc,0.845,0.842,0.853,0.849,0.776
auprc,0.014,0.013,0.016,0.011,0.005


Best threshold is 0.3959
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.733945,0.730609,0.740061,0.730612,0.607143


...

logistic_regression_median_quot.


/home/dwalke/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1165: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


,train_comp,train,val,test,gw
auroc,0.845,0.844,0.847,0.855,0.781
auprc,0.014,0.013,0.015,0.010,0.005


Best threshold is 0.3874
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.739187,0.738115,0.740061,0.742857,0.618304


In [45]:
prospective_pred_probas_mean_diff_ref[LogisticRegression.__name__] = lr_pred_proba["mean"]["diff."]["test"]

## XGBoost

In [46]:
##TODO Remove device
xgb_config = {
    "mean": {
        "diff.": {'alpha': 2.128588905051764, 'booster': 'dart', 'colsample_bytree': 0.6318855167964236, 'device': 'cuda:0', 'gamma': 1.4615950321058366, 'lambda': 0.007978256260982418, 'learning_rate': 0.07811625424993408, 'max_depth': 15, 'min_child_weight': 4.891040622635283, 'n_estimators': 250, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'scale_pos_weight': 1, 'subsample': 0.8583534678742919},
        "quot.": {'alpha': 0.0010403160060837718, 'booster': 'dart', 'colsample_bytree': 0.9879207850492944, 'device': 'cuda:0', 'gamma': 0.1355675325100431, 'lambda': 0.04607458541557399, 'learning_rate': 0.07510930339575483, 'max_depth': 5, 'min_child_weight': 7.1391668381518905, 'n_estimators': 200, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'scale_pos_weight': 1, 'subsample': 0.799240268324722}
    },
    "median": {
        "diff.": {'alpha': 0.010206058936634508, 'booster': 'gbtree', 'colsample_bytree': 0.9895062431392558, 'device': 'cuda:0', 'gamma': 3.676310313414377, 'lambda': 5.104786257926819, 'learning_rate': 0.0539413100508784, 'max_depth': 7, 'min_child_weight': 8.238728550335004, 'n_estimators': 800, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'scale_pos_weight': 1, 'subsample': 0.601842208870432},
        "quot.": {'alpha': 1.1505067020360504, 'booster': 'dart', 'colsample_bytree': 0.6384870100834997, 'device': 'cuda:0', 'gamma': 0.1515727934841835, 'lambda': 0.6781154502210275, 'learning_rate': 0.08154724036423781, 'max_depth': 6, 'min_child_weight': 5.275401094806807, 'n_estimators': 350, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'scale_pos_weight': 1, 'subsample': 0.9845815749162865}
    }
}

In [47]:
xgb_pred_proba = train_and_evaluate_models(XGBClassifier, xgb_config, "xgb", only_return_pred_proba= True)

xgb_mean_diff.


/home/dwalke/.local/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [10:20:09] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


,train_comp,train,val,test,gw
auroc,0.974,0.974,0.975,0.896,0.84
auprc,0.311,0.306,0.330,0.026,0.01


Best threshold is 0.0010999999940395355
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.974443,0.973311,0.978593,0.832653,0.725446


...

,train_comp,train,val,test,gw
sensitivity,0.922674,0.924937,0.914373,0.826531,0.707589


xgb_median_quot.


,train_comp,train,val,test,gw
auroc,0.974,0.973,0.977,0.893,0.836
auprc,0.291,0.286,0.308,0.026,0.010


Best threshold is 0.001500000013038516
Sensitivity for 80% specifity


,train_comp,train,val,test,gw
sensitivity,0.97772,0.975813,0.984709,0.836735,0.698661


In [48]:
prospective_pred_probas_mean_diff_ref[XGBClassifier.__name__] = xgb_pred_proba["mean"]["diff."]["test"]

In [50]:
prospective_pred_probas_mean_diff_ref.to_csv("pred_probas_prospective_mean_diff.csv", index = False)

In [72]:
y_true = prospective_pred_probas_mean_diff_ref["y"]
pred_proba = prospective_pred_probas_mean_diff_ref["RandomForestClassifier"]

In [73]:
from sklearn.metrics import roc_curve

min_sensitivity = .8
fpr, tpr, thresholds = roc_curve(y_true, pred_proba)
sensitivity_index = (tpr < min_sensitivity).sum()
thresholds[sensitivity_index]

0.39429656461226387

In [81]:
from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_true, pred_proba >= thresholds[sensitivity_index]).ravel()

In [82]:
tp / (tp + fn)

0.8

In [83]:
prospective_pred_probas_mean_diff_ref.columns

Index(['y', 'RandomForestClassifier', 'DecisionTreeClassifier',
       'LogisticRegression', 'XGBClassifier'],
      dtype='object')